# Automated Data Drift & Quality Monitoring Pipeline

## Overview

This notebook builds an **automated monitoring pipeline** using SageMaker Pipelines that:

- Runs a **SageMaker Processing Job** with Evidently to detect **data drift** and assess **data quality**
- Picks up the **latest CSV file** from baseline and production S3 paths
- Logs all metrics and HTML/JSON reports to a **SageMaker AI MLflow** experiment
- Sends **SNS email notifications** when data drift is detected
- Can be **scheduled via EventBridge** to run daily, weekly, or monthly

### Relationship to the Experimentation Notebook

This pipeline automates the monitoring workflow from Sections 6-8 of the companion notebook (`traditional_ml_experimentation_with_data_model_monitoring_evidently_v2.ipynb`):

| Notebook Section | Pipeline Automation |
|---|---|
| Section 6: Data Drift Monitoring | Processing Job runs `DataDriftPreset` |
| Section 8: Comprehensive Monitoring | Processing Job runs `DataSummaryPreset` |
| MLflow logging (metrics + artifacts) | Processing Job logs to same MLflow experiment |
| Manual execution | EventBridge scheduled trigger |

### Key Technologies

- **SageMaker Python SDK v3** - Simplified pipeline and processing APIs
- **SageMaker Pipelines** - Managed ML workflow orchestration
- **SageMaker Processing** - Managed compute for monitoring workloads
- **SageMaker AI MLflow** - Experiment tracking and artifact storage
- **Evidently** - Open-source data drift and quality monitoring
- **Amazon EventBridge** - Scheduled pipeline triggers
- **Amazon SNS** - Email notifications for drift alerts

---

## Section 1: Prerequisites & Setup

Install required packages and import libraries.

In [ ]:
# Install required packages
%pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops
%pip install --no-cache-dir sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops 'sagemaker>=3' mlflow==3.4.0 sagemaker-mlflow==0.2.0

In [ ]:
%pip show sagemaker

In [ ]:
# Import required libraries
import boto3
import sagemaker
import json
import time
from datetime import datetime

# SageMaker SDK v3 imports
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris
from sagemaker.core.transformer import Transformer

# SageMaker Processing imports (v3 paths)
from sagemaker.core.processing import ScriptProcessor
from sagemaker.core.shapes import (
    ProcessingInput, ProcessingS3Input,
    ProcessingOutput, ProcessingS3Output,
)

# SageMaker Pipeline imports (v3 paths)
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.mlops.workflow.steps import ProcessingStep, TransformStep
from sagemaker.core.workflow.parameters import ParameterString

In [ ]:
# Initialize SageMaker session and AWS configuration
sagemaker_session = Session()
pipeline_session = PipelineSession()
region = sagemaker_session.boto_region_name
role = get_execution_role()
bucket = sagemaker_session.default_bucket()
prefix = 'monitoring-pipeline'
timestamp_suffix = datetime.now().strftime('%Y%m%d%H%M%S')

sm_client = boto3.client('sagemaker', region_name=region)
sns_client = boto3.client('sns', region_name=region)
scheduler_client = boto3.client('scheduler', region_name=region) 
iam_client = boto3.client('iam', region_name=region)

print(f'Region: {region}')
print(f'S3 Bucket: {bucket}')
print(f'IAM Role: {role}')
print(f'timestamp_suffix: {timestamp_suffix}')

---

## Section 2: Configuration

Set the high-level inputs for the monitoring pipeline. Update these values to match your environment.

In [ ]:
# ============================================================
# UPDATE THESE VALUES FOR YOUR ENVIRONMENT
# ============================================================

# --- Baseline Data Configuration ---
# S3 path to your baseline/reference data (training data distribution)
# This represents the "expected" data distribution against which drift is measured
baseline_filename = 'baseline.csv'
baseline_data_file_prefix = 'data/baseline/baseline.csv'
baseline_data_prefix = 'data/baseline'
baseline_s3_uri = f's3://{bucket}/{prefix}/{baseline_data_file_prefix}'

# --- Input Data Configuration ---
# Two versions of the same data are needed:
# 1. WITHOUT headers: Used by batch transform for inference (model expects no headers)
# 2. WITH headers: Used by Evidently for drift analysis (needs column names)
input_filename = 'test_features.csv'  # Without headers for batch transform
input_data_file_prefix = 'data/test/test_features.csv'
input_data_folder_prefix = 'data/test'
input_s3_uri = f's3://{bucket}/{prefix}/{input_data_file_prefix}'

input_headers_filename = 'test_features_headers.csv'  # WITH headers for Evidently
input_headers_data_file_prefix = 'data/test/test_features_headers.csv'
input_headers_s3_uri = f's3://{bucket}/{prefix}/{input_headers_data_file_prefix}'

# --- Inference Output Configuration ---
# S3 path where batch transform will write predictions
inference_data_prefix = 'data/inference'
production_s3_uri = f's3://{bucket}/{prefix}/{inference_data_prefix}'

# --- MLflow Configuration ---
# IMPORTANT: Use the SAME MLflow app and experiment as the training notebook
# This ensures unified experiment tracking across training and monitoring
mlflow_app_name = 'DefaultMLFlowApp'
mlflow_experiment_name = 'test-monitor-pipeline'  # SAME as training notebook
mlflow_run_name = f"data_drift_quality_monitoring_{timestamp_suffix}"

# --- Alerting Configuration ---
# Email address to receive drift alert notifications
# You MUST confirm the SNS subscription in your inbox after running Section 4
notification_email = '<ENTER>'  # REPLACE with your email

# --- Scheduling Configuration ---
# How often the pipeline should run automatically
# Examples: 'rate(1 day)', 'rate(6 hours)', 'cron(0 8 * * ? *)'
schedule_expression = 'rate(1 day)'

# --- Pipeline Infrastructure Configuration ---
pipeline_name = 'data-drift-monitoring-pipeline'
instance_type = 'ml.m5.large'  # Instance type for processing job

# --- Model Configuration ---
# S3 URI to your trained model artifacts (from training notebook)
# Get this from: model_trainer._latest_training_job.model_artifacts.s3_model_artifacts
model_data_s3_uri = '<ENTER>'  # REPLACE with your model S3 URI
sagemaker_model_name = f'xgb-{prefix}-{timestamp_suffix}'

# --- Print Configuration Summary ---
print(f'  Baseline S3 URI     : {baseline_s3_uri}')
print(f'  Production S3 URI   : {production_s3_uri}')
print(f'  MLflow App          : {mlflow_app_name}')
print(f'  MLflow Experiment   : {mlflow_experiment_name}')
print(f'  MLflow Run Name     : {mlflow_run_name}')
print(f'  Notification Email  : {notification_email}')
print(f'  Schedule            : {schedule_expression}')
print(f'  Pipeline Name       : {pipeline_name}')
print(f'  SM model S3 URI     : {model_data_s3_uri}')
print(f'  Sagemaker Model Name: {sagemaker_model_name}')
print(f'  Input S3 URI        : {input_s3_uri}')

In [ ]:
# Stage baseline data to S3
# This uploads your reference/training data that Evidently will use as the baseline
# for drift detection. Ensure this file has column headers.
baseline_data_s3 = sagemaker_session.upload_data(
    'data/baseline.csv', 
    bucket, 
    f'{prefix}/{baseline_data_prefix}'
)

print(f'Baseline data uploaded to: {baseline_data_s3}')

In [ ]:
# Stage input data to S3
# Uploads TWO versions of the same data:
# 1. WITHOUT headers (test_features.csv): For batch transform inference
# 2. WITH headers (test_features_headers.csv): For Evidently drift analysis
#
# Why two files? Batch Transform expects data without headers, but Evidently
# needs headers to match feature names between baseline and current data.
input_data_s3 = sagemaker_session.upload_data(
    'data/test_features.csv',  # Without headers
    bucket, 
    f'{prefix}/{input_data_folder_prefix}'
)

input_headers_data_s3 = sagemaker_session.upload_data(
    'data/test_features_headers.csv',  # With headers
    bucket, 
    f'{prefix}/{input_data_folder_prefix}'
)

print(f'Input data uploaded to: {input_data_s3}, {input_headers_data_s3}')

---

## Section 3: SageMakerAI MLflow App ARN

Look up the SageMaker AI MLflow App ARN from the app name.

In [ ]:
# Resolve MLflow App ARN
mlflow_list = sm_client.list_mlflow_apps()

mlflow_app_arn = None
for app in mlflow_list['Summaries']:
    if app['Name'] == mlflow_app_name:
        
        mlflow_app_arn = app['Arn']
        break

if mlflow_app_arn:
    print(f'MLflow App: {mlflow_app_name}')
    print(f'ARN: {mlflow_app_arn}')
else:
    raise ValueError(
        f'MLflow app "{mlflow_app_name}" not found. '
        f'Available apps: {[a["Name"] for a in mlflow_list["Summaries"]]}'
    )

---

## Section 4: Create SNS Topic & Email Subscription

Create an SNS topic for drift alert notifications and subscribe the configured email address. After running this cell, **check your inbox and confirm the SNS subscription** before drift alerts can be delivered.

In [ ]:
# Create SNS topic for drift alerts
topic_name = f'{pipeline_name}-drift-alerts'

create_topic_response = sns_client.create_topic(
    Name=topic_name,
    Tags=[
        {'Key': 'Project', 'Value': 'sagemaker-batch-monitoring'},
        {'Key': 'ManagedBy', 'Value': 'monitoring-pipeline-notebook'},
    ]
)
sns_topic_arn = create_topic_response['TopicArn']
print(f'SNS Topic created: {sns_topic_arn}')

# Subscribe email address
if notification_email and notification_email != 'your-email@example.com':
    subscribe_response = sns_client.subscribe(
        TopicArn=sns_topic_arn,
        Protocol='email',
        Endpoint=notification_email,
    )
    print(f'Email subscription requested for: {notification_email}')
    print(f'Subscription ARN: {subscribe_response["SubscriptionArn"]}')
    print()
    print('*** IMPORTANT: Check your inbox and confirm the SNS subscription ***')
    print('*** Drift alerts will NOT be delivered until the subscription is confirmed ***')
else:
    print('WARNING: notification_email is not set. Update it in Section 2 and re-run this cell.')
    print(f'SNS Topic ARN (subscribe manually later): {sns_topic_arn}')

---

## Section 5: Prepare Batch Transform for Inference

Before creating the monitoring pipeline, we need to set up the model for batch inference.

### Why Batch Transform?
- **No Endpoint Costs**: Only pay for compute during inference jobs
- **Scalable**: Process large datasets efficiently
- **Integrated**: Output feeds directly into monitoring step

### Steps:
1. Get the XGBoost inference container image
2. Create a SageMaker Model from trained artifacts
3. Configure for use in the pipeline

In [ ]:
# Get inference image URI
inference_image = image_uris.retrieve(
    framework='xgboost',
    region=region,
    version='1.7-1',
    image_scope='inference'
)
print(f'XGBoost inference image: {inference_image}')

In [ ]:
# Create SageMaker Model

create_model_response = sm_client.create_model(
    ModelName=sagemaker_model_name,
    PrimaryContainer={
        'Image': inference_image,
        'ModelDataUrl': model_data_s3_uri
    },
    ExecutionRoleArn=role
)

print(f'Model created: {sagemaker_model_name}')

---

## Section 6: Create the SageMaker Monitoring Pipeline

### Details

```
EventBridge trigger (scheduled)

SageMakerAI Pipeline with Batch Transform and Processing Step.

Processing Step (ml.m5.xlarge)
  - scripts/monitoring_processor.py
  - Loads latest CSV from baseline & production S3 paths
  - Runs Evidently DataDriftPreset + DataSummaryPreset
  - Logs metrics & reports to MLflow
  - Sends SNS alert if drift detected
  - Saves reports to S3 output
```

### Pipeline Parameters

The pipeline accepts runtime parameters so you can override S3 paths and configuration without modifying the pipeline definition.

In [ ]:
# Define pipeline parameters (can be overridden at execution time)
# These parameters make the pipeline flexible - you can change data sources
# without rebuilding the entire pipeline definition

param_baseline_s3_uri = ParameterString(
    name='BaselineS3Uri',
    default_value=baseline_s3_uri,
)

param_input_s3_uri = ParameterString(
    name='InputS3Uri',
    default_value=input_s3_uri,
)

param_input_headers_s3_uri = ParameterString(
    name='InputHeadersS3Uri',
    default_value=input_headers_s3_uri,
)



In [ ]:
# Define the Batch Transform Step for the SageMaker Pipeline
# This step generates predictions on the input data before monitoring

transformer = Transformer(
    model_name=sagemaker_model_name,  # The trained XGBoost model
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=production_s3_uri,  # Where predictions will be saved
    sagemaker_session=pipeline_session,
)

# Configure the transform job arguments
transform_args = transformer.transform(
    data=param_input_s3_uri,  # Input data WITHOUT headers
    content_type='text/csv',
    split_type='Line',  # Process line-by-line
)

# Create the pipeline step
transform_step = TransformStep(
    name='BatchTransform',
    step_args=transform_args,
)

# Note: The processing step will depend on this step's output
# through transform_step.properties.TransformOutput.S3OutputPath

In [ ]:
# Get the SKLearn processing image URI
sklearn_image_uri = image_uris.retrieve(
    framework='sklearn',
    region=region,
    version='1.2-1',
    py_version='py3',
    instance_type=instance_type,
)
print(f'Processing image: {sklearn_image_uri}')

In [ ]:
# Create the ScriptProcessor for the monitoring job
monitoring_processor = ScriptProcessor(
    role=role,
    image_uri=sklearn_image_uri,
    command=['python3'],
    instance_count=1,
    instance_type=instance_type,
    volume_size_in_gb=30,
    max_runtime_in_seconds=3600,
    base_job_name='evidently-monitoring',
    sagemaker_session=pipeline_session,
    env={
        # MLflow tracking URI is also passed as env var for the sagemaker-mlflow plugin
        'MLFLOW_TRACKING_URI': mlflow_app_arn,
    },
)

print(f'ScriptProcessor configured with {instance_type}')

In [ ]:
# Define the Processing Step for the SageMaker Pipeline
# This step runs the Evidently monitoring script on managed compute

processing_args = monitoring_processor.run(
    code='scripts/monitoring_processor.py',
    
    # --- INPUT CHANNELS ---
    # SageMaker mounts these S3 locations to local paths in the container
    inputs=[
        # Channel 1: Baseline/reference data (training distribution)
        ProcessingInput(
            input_name='baseline_data',
            s3_input=ProcessingS3Input(
                s3_uri=param_baseline_s3_uri,  # Uses pipeline parameter
                local_path='/opt/ml/processing/baseline',
                s3_data_type='S3Prefix',
                s3_input_mode='File',
            ),
        ),
        # Channel 2: Current input data WITHOUT headers (for batch transform)
        ProcessingInput(
            input_name='input_data',
            s3_input=ProcessingS3Input(
                s3_uri=param_input_s3_uri,  # Uses pipeline parameter
                local_path='/opt/ml/processing/current',
                s3_data_type='S3Prefix',
                s3_input_mode='File',
            ),
        ),
        # Channel 3: Predictions from the batch transform step
        # Uses step properties to create dependency between steps
        ProcessingInput(
            input_name='predictions_data',
            s3_input=ProcessingS3Input(
                s3_uri=transform_step.properties.TransformOutput.S3OutputPath,
                local_path='/opt/ml/processing/predictions',
                s3_data_type='S3Prefix',
                s3_input_mode='File',
            ),
        ),
        # Channel 4: Current input data WITH headers (for Evidently analysis)
        # This is the same data as Channel 2 but includes column names
        ProcessingInput(
            input_name='input_headers_data',
            s3_input=ProcessingS3Input(
                s3_uri=param_input_headers_s3_uri,  # Uses pipeline parameter
                local_path='/opt/ml/processing/current_headers',
                s3_data_type='S3Prefix',
                s3_input_mode='File',
            ),
        ),
    ],
    
    # --- OUTPUT CHANNEL ---
    # Where monitoring reports and artifacts will be saved
    outputs=[
        ProcessingOutput(
            output_name='monitoring-reports',
            s3_output=ProcessingS3Output(
                s3_uri=f's3://{bucket}/{prefix}/monitoring-reports',
                local_path='/opt/ml/processing/output',
                s3_upload_mode='EndOfJob',  # Upload after job completes
            ),
        ),
    ],
    
    # --- SCRIPT ARGUMENTS ---
    # Command-line arguments passed to monitoring_processor.py
    arguments=[
        '--baseline-filename', baseline_filename,
        '--input-filename', input_filename,
        '--input-headers-filename', input_headers_filename,
        '--mlflow-tracking-uri', mlflow_app_arn,
        '--mlflow-experiment-name', mlflow_experiment_name,
        '--mlflow-run-name', mlflow_run_name,
        '--sns-topic-arn', sns_topic_arn,
        '--region', region,
    ],
    
)

In [ ]:
processing_step = ProcessingStep(
    name='EvidentlyMonitoringStep',
    step_args=processing_args,
)

print('ProcessingStep defined: EvidentlyMonitoringStep')

In [ ]:
# Create the SageMaker Pipeline
monitoring_pipeline = Pipeline(
    name=pipeline_name,
    parameters=[param_baseline_s3_uri, param_input_s3_uri, param_input_headers_s3_uri],
    steps=[transform_step, processing_step],
    sagemaker_session=pipeline_session,
)

print(f'Pipeline "{pipeline_name}" defined with {len(monitoring_pipeline.steps)} step(s)')

In [ ]:
# Create or update the pipeline in SageMaker
monitoring_pipeline.upsert(role_arn=role)
print(f'Pipeline "{pipeline_name}" created/updated successfully')

---

## Section 7: Test the Pipeline

Execute the pipeline once to verify everything works end-to-end. This will launch a SageMaker Processing Job that runs the Evidently monitoring and logs results to MLflow.

In [ ]:
%%time
# Execute the pipeline (test run)
execution = monitoring_pipeline.start()

print(f'Pipeline execution started: {execution.arn}')
print('Waiting for pipeline execution to complete...')
print('You can monitor progress in SageMaker Studio under Pipelines.')

execution.wait()

print(f'\nPipeline execution status: {execution.describe()["PipelineExecutionStatus"]}')

In [ ]:
# View execution details
execution_steps = execution.list_steps()
for step in execution_steps:
    print(f'Step: {step["StepName"]}')
    print(f'  Status: {step["StepStatus"]}')
    if 'FailureReason' in step:
        print(f'  Failure Reason: {step["FailureReason"]}')

### View Results in MLflow

After the pipeline completes successfully:

**Step 1:** Open SageMaker Studio and navigate to the **MLflow** panel

**Step 2:** Find the experiment: **test-monitor-pipeline** (same as training notebook)

**Step 3:** Look for the run named `data_drift_quality_monitoring_<timestamp>`

**Step 4:** View the following:
- **Metrics Tab**:
  - `DriftedColumnsCount.count`: Number of features with drift
  - `DriftedColumnsCount.share`: Percentage of features with drift
  - `ValueDrift:*`: Per-feature drift scores (only for drifted features)
  
- **Artifacts Tab**:
  - `evidently_report_data_drift_html`: Interactive HTML drift report
  - `evidently_report_data_drift_json`: Structured JSON for programmatic access
  - `evidently_report_data_quality`: HTML data quality assessment
  
- **Parameters Tab**:
  - Data sources, row counts, monitoring timestamp

**Benefits of MLflow Integration:**
- Track drift trends over time by comparing runs
- Correlate drift with model performance changes
- Unified view of training and monitoring in same experiment

---

## Section 8: Schedule the Pipeline with EventBridge Scheduler

Create an EventBridge Scheduler schedule that triggers the pipeline on a recurring schedule. This uses the **new AWS EventBridge Scheduler API** (not the legacy EventBridge Rules).

### How it works

1. EventBridge Scheduler fires on the configured schedule (e.g., daily at midnight)
2. Scheduler assumes an IAM role to call `sagemaker:StartPipelineExecution`
3. SageMaker starts the pipeline with the default parameter values
4. The processing job runs Evidently monitoring and logs to MLflow
5. If drift is detected, SNS sends an email alert

### Key Differences from Legacy EventBridge Rules

- Uses `scheduler.amazonaws.com` service principal (not `events.amazonaws.com`)
- Single API call `create_schedule()` instead of `put_rule()` + `put_targets()`
- Requires `FlexibleTimeWindow` parameter
- Better native support for SageMaker Pipeline parameters

In [ ]:
# Get the pipeline ARN
pipeline_description = sm_client.describe_pipeline(PipelineName=pipeline_name)
pipeline_arn = pipeline_description['PipelineArn']
print(f'Pipeline ARN: {pipeline_arn}')

In [ ]:
# Create IAM role for EventBridge Scheduler to invoke the SageMaker Pipeline
scheduler_role_name = f'{pipeline_name}-scheduler-role'
account_id = boto3.client('sts').get_caller_identity()['Account']

# IMPORTANT: Must use scheduler.amazonaws.com (not events.amazonaws.com)
trust_policy = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Effect': 'Allow',
            'Principal': {'Service': 'scheduler.amazonaws.com'},  # EventBridge Scheduler service
            'Action': 'sts:AssumeRole',
        }
    ],
}

# Create or update the role
try:
    # Try to update the trust policy first (in case role exists with wrong trust policy)
    iam_client.update_assume_role_policy(
        RoleName=scheduler_role_name,
        PolicyDocument=json.dumps(trust_policy)
    )
    print(f'IAM Role trust policy updated: {scheduler_role_name}')
except iam_client.exceptions.NoSuchEntityException:
    # Role doesn't exist, create it
    iam_client.create_role(
        RoleName=scheduler_role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='Allows EventBridge Scheduler to start SageMaker Pipeline executions',
        Tags=[
            {'Key': 'Project', 'Value': 'sagemaker-batch-monitoring'},
            {'Key': 'ManagedBy', 'Value': 'monitoring-pipeline-notebook'},
        ],
    )
    print(f'IAM Role created: {scheduler_role_name}')

# Attach inline policy to allow starting the pipeline
policy_document = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Effect': 'Allow',
            'Action': 'sagemaker:StartPipelineExecution',
            'Resource': pipeline_arn,
        }
    ],
}

iam_client.put_role_policy(
    RoleName=scheduler_role_name,
    PolicyName='StartPipelineExecution',
    PolicyDocument=json.dumps(policy_document),
)

scheduler_role_arn = f'arn:aws:iam::{account_id}:role/{scheduler_role_name}'
print(f'EventBridge Scheduler Role ARN: {scheduler_role_arn}')
print(f'Policy attached: StartPipelineExecution')

# Allow IAM role propagation
print('Waiting for IAM role to propagate...')
time.sleep(10)

In [ ]:
# Create EventBridge Scheduler schedule (not legacy EventBridge Rule)
schedule_name = f'{pipeline_name}-auto-schedule'

# Prepare pipeline parameters for the target
pipeline_parameters = [
    {'Name': 'BaselineS3Uri', 'Value': baseline_s3_uri},
    {'Name': 'InputS3Uri', 'Value': input_s3_uri},
    {'Name': 'InputHeadersS3Uri', 'Value': input_headers_s3_uri},
]

try:
    # Try to update schedule if it exists
    scheduler_client.update_schedule(
        Name=schedule_name,
        ScheduleExpression=schedule_expression,
        State='ENABLED',
        FlexibleTimeWindow={
            'Mode': 'OFF'  # Execute at the exact scheduled time
        },
        Target={
            'Arn': pipeline_arn,
            'RoleArn': scheduler_role_arn,
            'SageMakerPipelineParameters': {
                'PipelineParameterList': pipeline_parameters
            }
        },
    )
    print(f'EventBridge Scheduler schedule updated: {schedule_name}')
except scheduler_client.exceptions.ResourceNotFoundException:
    # Schedule doesn't exist, create it
    scheduler_client.create_schedule(
        Name=schedule_name,
        Description=f'Triggers {pipeline_name} on schedule: {schedule_expression}',
        ScheduleExpression=schedule_expression,
        State='ENABLED',
        FlexibleTimeWindow={
            'Mode': 'OFF'  # Execute at the exact scheduled time
        },
        Target={
            'Arn': pipeline_arn,
            'RoleArn': scheduler_role_arn,
            'SageMakerPipelineParameters': {
                'PipelineParameterList': pipeline_parameters
            }
        },
    )
    print(f'EventBridge Scheduler schedule created: {schedule_name}')

print(f'Schedule Expression: {schedule_expression}')
print(f'\n✅ EventBridge Scheduler configured successfully!')
print(f'The pipeline will run automatically on: {schedule_expression}')
print(f'Schedule name: {schedule_name}')

---

## Section 8: Verify Schedule

Confirm the EventBridge rule and target are properly configured.

In [ ]:
# Verify the EventBridge Scheduler schedule
schedule_info = scheduler_client.get_schedule(Name=schedule_name)

print(f'Schedule: {schedule_info["Name"]}')
print(f'  State: {schedule_info["State"]}')
print(f'  Schedule Expression: {schedule_info["ScheduleExpression"]}')
print(f'  ARN: {schedule_info["Arn"]}')
print(f'\nTarget Configuration:')
print(f'  Pipeline ARN: {schedule_info["Target"]["Arn"]}')
print(f'  Role ARN: {schedule_info["Target"]["RoleArn"]}')
print(f'\nPipeline Parameters:')
for param in schedule_info["Target"]["SageMakerPipelineParameters"]["PipelineParameterList"]:
    print(f'  - {param["Name"]}: {param["Value"]}')

---

## Section 9: Summary

### What was created

| Resource | Name | Purpose |
|---|---|---|
| **SageMaker Pipeline** | `data-drift-monitoring-pipeline` | Orchestrates the monitoring processing job |
| **Processing Step** | `EvidentlyMonitoringStep` | Runs Evidently data drift + quality reports |
| **SNS Topic** | `data-drift-monitoring-pipeline-drift-alerts` | Email notifications for drift alerts |
| **EventBridge Scheduler Schedule** | `data-drift-monitoring-pipeline-auto-schedule` | Triggers pipeline on schedule (NEW API) |
| **IAM Role** | `data-drift-monitoring-pipeline-scheduler-role` | Allows EventBridge Scheduler to start the pipeline |

### What happens on each scheduled run

1. **EventBridge Scheduler** triggers the pipeline at the configured interval
2. A SageMaker Processing Job starts on `ml.m5.large`
3. The processing script picks up the **latest CSV** from baseline and production S3 paths
4. **Evidently** generates `DataDriftPreset` and `DataSummaryPreset` reports
5. Metrics and HTML/JSON report artifacts are logged to the **MLflow** experiment
6. If data drift is detected, an **email alert** is sent via SNS
7. Reports are also saved to `s3://<bucket>/monitoring-pipeline/monitoring-reports/`

### Schedule examples

| Schedule | Expression |
|---|---|
| Once per day | `rate(1 day)` |
| Every 6 hours | `rate(6 hours)` |
| 8 AM on 1st of each month | `cron(0 8 1 * ? *)` |
| 6 AM every Monday | `cron(0 6 ? * MON *)` |

To change the schedule, update `schedule_expression` in Section 2 and re-run Section 7.

### EventBridge Scheduler vs Legacy EventBridge Rules

This notebook uses the **new EventBridge Scheduler** API which provides:
- Better reliability and scalability
- Native support for SageMaker Pipeline parameters
- Simplified API (single call vs. multiple)
- Required for new AWS deployments


## Section 11: Cleanup (Optional)

Run the cells below to remove all resources created by this notebook.

**Note:** MLflow experiment data and S3 artifacts are preserved for historical analysis and auditing.

In [ ]:
# Remove EventBridge Scheduler schedule
try:
    scheduler_client.delete_schedule(Name=schedule_name)
    print(f'Deleted EventBridge Scheduler schedule: {schedule_name}')
except Exception as e:
    print(f'Error deleting EventBridge Scheduler schedule: {e}')

# Delete SageMaker Pipeline
try:
    sm_client.delete_pipeline(PipelineName=pipeline_name)
    print(f'Deleted pipeline: {pipeline_name}')
except Exception as e:
    print(f'Error deleting pipeline: {e}')

# Delete SNS topic (also removes subscriptions)
try:
    sns_client.delete_topic(TopicArn=sns_topic_arn)
    print(f'Deleted SNS topic: {sns_topic_arn}')
except Exception as e:
    print(f'Error deleting SNS topic: {e}')

# Delete IAM role
try:
    iam_client.delete_role_policy(RoleName=scheduler_role_name, PolicyName='StartPipelineExecution')
    iam_client.delete_role(RoleName=scheduler_role_name)
    print(f'Deleted IAM role: {scheduler_role_name}')
except Exception as e:
    print(f'Error deleting IAM role: {e}')

print('\nCleanup complete. MLflow runs and S3 data are preserved.')